# Connect-sum RL experiments (persistent local setup)

This notebook is a persistent experiment variant of the dependency notebook. It is focused on connect-sum experiments and analysis, not dependency propagation.

## What this notebook does

1. Reuses a persistent workbook copy at `outputs/unknotting_connect_sum_experiments.xlsx` (created once).
2. Hardcodes base knots: trefoil `3_1` and both 5-crossing knots `5_1`, `5_2`.
3. Builds experiment families:
   - `3_1 # (all exact 10-crossing knots)`
   - `(5_1, 5_2) # (all exact 8-crossing knots)`
4. Flips one crossing, keeps configurable 13->13 cases, runs RL reduction, and identifies reduced knots in the workbook.
5. Writes per-trial JSONL + run summary JSON + run manifest JSON under `outputs/`.

## Start a run: local requirements

- Python 3.10+
- Virtual environment with `pip install -r requirements.txt`
- Jupyter Lab/Notebook
- `data/unknotting.xlsx` present
- Model file present in one of: `models/best_model.zip`, `models/ppo_knot_rl_spherogram_continued.zip`, `outputs/best_model.zip`

## Optional GCP setup (only if you use cloud training/data paths)

This repository can run fully local; GCP is optional.

If you do use GCP-backed paths, ensure:

- A Google Cloud project exists and billing is enabled.
- Cloud Storage API is enabled.
- You have bucket read access (typically Storage Object Viewer).
- `gcloud auth login` has been run for CLI workflows.
- `gcloud auth application-default login` has been run for ADC-based Python auth.
- `GOOGLE_CLOUD_PROJECT` is set when needed by your environment or helper code.


In [ ]:
# Local setup (run once per environment, if needed)
# %pip install -r requirements.txt

In [ ]:
import os, re, json, ast, math, random, csv, glob, time, shutil
from pathlib import Path
from dataclasses import dataclass
from fractions import Fraction
from collections import defaultdict
from typing import Iterable, Optional, List, Tuple, Dict, Any

import numpy as np
import pandas as pd

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

from tqdm.auto import tqdm

import snappy
from spherogram import Link

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


In [ ]:
# Local repository paths (no Google Drive needed)
from pathlib import Path

CANDIDATE_ROOTS = [Path.cwd(), Path.cwd().parent]
REPO_ROOT = None
for cand in CANDIDATE_ROOTS:
    if (cand / "data").exists() or (cand / "models").exists() or (cand / "training_data").exists():
        REPO_ROOT = cand
        break
if REPO_ROOT is None:
    REPO_ROOT = Path.cwd()

DATA_DIR = REPO_ROOT / "data"
MODELS_DIR = REPO_ROOT / "models"
TRAINING_DIR = REPO_ROOT / "training_data"
OUT_DIR = REPO_ROOT / "outputs"

for folder in [DATA_DIR, MODELS_DIR, TRAINING_DIR, OUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

SOURCE_XLSX_PATH = DATA_DIR / "unknotting.xlsx"
if not SOURCE_XLSX_PATH.exists():
    raise FileNotFoundError(
        f"Missing workbook: {SOURCE_XLSX_PATH}\n"
        "Place your database at repo-root/data/unknotting.xlsx"
    )

# Persistent one-time workbook copy for connect-sum experiments.
EXPERIMENT_XLSX_PATH = OUT_DIR / "unknotting_connect_sum_experiments.xlsx"
if not EXPERIMENT_XLSX_PATH.exists():
    shutil.copy2(SOURCE_XLSX_PATH, EXPERIMENT_XLSX_PATH)
    print("Created workbook copy:", EXPERIMENT_XLSX_PATH)
else:
    print("Reusing workbook copy:", EXPERIMENT_XLSX_PATH)

XLSX_PATH = EXPERIMENT_XLSX_PATH
BASE = REPO_ROOT

print("REPO_ROOT:", REPO_ROOT)
print("Source   :", SOURCE_XLSX_PATH)
print("Workbook :", XLSX_PATH)
print("MODELS   :", MODELS_DIR)
print("TRAINING :", TRAINING_DIR)
print("OUTPUTS  :", OUT_DIR)


In [ ]:
# ----------------------------
# Configuration
# ----------------------------

# Hardcoded experiment anchors
HARD_TREFOIL_KNOT_ID = "3_1"
HARD_FIVE_CROSSING_KNOT_IDS = ["5_1", "5_2"]

# Experiment families
RUN_TREFOIL_WITH_10_EXACT = True
RUN_FIVE_WITH_8_EXACT = True

# Crossing filters for the connect-sum study
REQUIRE_PREFLIP_13 = True
REQUIRE_POSTFLIP_13 = True

# Runtime limits
MAX_CASES_PER_FAMILY = None   # set int for smoke testing
MAX_FLIPS_PER_CASE = None     # set int to cap number of flipped crossings per case
NUM_VARIANTS_PER_KNOT = 1     # connect-sum trials use direct PD; keep 1 unless experimenting

# RL settings
UNKNOTTER_EPISODES_PER_FLIP = 1
UNKNOTTER_MAX_STEPS = 500
TRAIN_IF_MODEL_MISSING = True
TRAIN_STEPS_IF_NEEDED = 20000

# Keep original move settings available for helper compatibility
BACKTRACK_STEPS_MIN = 6
BACKTRACK_STEPS_MAX = 8
RIII_STEPS_MAX = 20

# Identification window
MIN_DATABASE_CROSSINGS = 1
MAX_DATABASE_CROSSINGS = 13

# Output behavior (analysis-first by default)
WRITE_UPDATED_WORKBOOK = False
SAVE_EVERY_KNOT = False
OVERWRITE_EXPERIMENT_OUTPUTS = True

# Dependency propagation controls
ENABLE_DEPENDENCY_PROPAGATION = True
MAX_PROPAGATION_PASSES = None
OVERWRITE_DEPENDENCY_OUTPUTS = True
MAX_DEPENDENCY_EVIDENCE_PER_KNOT = 5

RUN_STAMP = time.strftime("%Y%m%d-%H%M%S")
RESULTS_JSONL_PATH = OUT_DIR / "connect_sum_experiment_results.jsonl"
SUMMARY_JSON_PATH = OUT_DIR / "connect_sum_experiment_summary.json"
RUN_MANIFEST_PATH = OUT_DIR / "connect_sum_experiment_manifest.json"
ALL_DEPENDENCIES_JSON_PATH = OUT_DIR / "connect_sum_dependency_all.json"
UNIMPROVED_DEPENDENCIES_JSON_PATH = OUT_DIR / "connect_sum_dependency_unimproved.json"
PROPAGATION_RESULTS_JSON_PATH = OUT_DIR / "connect_sum_dependency_propagation_results.json"

MODEL_PATH_CANDIDATES = [
    BASE / "best_model.zip",
    BASE / "ppo_knot_rl_spherogram_continued.zip",
    OUT_DIR / "best_model.zip",
]

# Training data sources from the original notebook / paper pipeline
GCS_CSV_PATH_MAIN = "gs://gdm-unknotting/hard_unknots.csv"
GCS_CSV_PATH_VERY = "gs://gdm-unknotting/very_hard_unknots.csv"

# Optional local extras: only used if present
LOCAL_EXTRA_FILES = [
    BASE / "random_diagrams.csv",
    BASE / "random_diagrams.txt",
    BASE / "hard_unknots.csv",
    BASE / "very_hard_unknots.csv",
]

# Parallelization settings kept for compatibility
ENABLE_PARALLEL = False
PARALLEL_MAX_WORKERS = None
PARALLEL_RESERVE_CORES = 1
PARALLEL_INFERENCE_DEVICE = "cpu"


In [ ]:
# ----------------------------
# Helpers: workbook / parsing
# ----------------------------
_int_pat = re.compile(r'-?\d+')

def pick_first_existing(df, candidates):
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None

def parse_pd_cell(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return None
    if isinstance(x, list):
        return [[int(y) for y in q] for q in x]
    s = str(x).strip()
    if not s or s.lower() in {"nan", "none"}:
        return None
    try:
        obj = ast.literal_eval(s)
        if isinstance(obj, list) and all(isinstance(q, (list, tuple)) and len(q) == 4 for q in obj):
            return [[int(y) for y in q] for q in obj]
    except Exception:
        pass
    items = re.findall(r'[Xx]\s*\[([^\]]+)\]', s)
    if items:
        out = []
        for it in items:
            nums = [int(z.strip()) for z in it.split(',')]
            if len(nums) != 4:
                return None
            out.append(nums)
        return out
    return None

def parse_vector_cell(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return None
    if isinstance(x, (list, tuple)):
        try:
            return [int(v) for v in x]
        except Exception:
            return None
    s = str(x).strip()
    if not s or s.lower() in {"nan", "none"}:
        return None
    if s.startswith("[") and s.endswith("]"):
        try:
            v = ast.literal_eval(s)
            if isinstance(v, (list, tuple)):
                return [int(z) for z in v]
        except Exception:
            pass
    nums = _int_pat.findall(s)
    if not nums:
        return None
    return [int(z) for z in nums]

def ensure_minmax_coeffs(v):
    if v is None:
        return None
    v = [int(x) for x in v]
    if len(v) < 3:
        return None
    mn, mx = v[0], v[1]
    coeffs = v[2:]
    if len(coeffs) != abs(mx - mn) + 1:
        return None
    return mn, mx, coeffs

def strip_leading_trailing_zeros(coeffs):
    coeffs = list(map(int, coeffs))
    i, j = 0, len(coeffs)
    while i < j and coeffs[i] == 0:
        i += 1
    while j > i and coeffs[j-1] == 0:
        j -= 1
    out = coeffs[i:j]
    return out if out else [0]

def canon_coeff_key(coeffs):
    return tuple(strip_leading_trailing_zeros(coeffs))

def canon_coeff_key_mirror(coeffs):
    return tuple(reversed(strip_leading_trailing_zeros(coeffs)))

def span_abs(mn, mx):
    return abs(int(mx) - int(mn))

def parse_unknotting_entry(x):
    """
    Returns dict with:
      kind: 'missing' | 'exact' | 'range' | 'other'
      lower, upper: ints or None
    """
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return {"kind": "missing", "lower": None, "upper": None, "raw": x}
    if isinstance(x, (int, np.integer)):
        n = int(x)
        return {"kind": "exact", "lower": n, "upper": n, "raw": x}
    if isinstance(x, float) and float(x).is_integer():
        n = int(x)
        return {"kind": "exact", "lower": n, "upper": n, "raw": x}

    s = str(x).strip()
    if not s or s.lower() in {"nan", "none"}:
        return {"kind": "missing", "lower": None, "upper": None, "raw": x}

    try:
        obj = ast.literal_eval(s)
        if isinstance(obj, (list, tuple)) and len(obj) == 2:
            a, b = int(obj[0]), int(obj[1])
            if a == b:
                return {"kind": "exact", "lower": a, "upper": b, "raw": x}
            return {"kind": "range", "lower": min(a,b), "upper": max(a,b), "raw": x}
    except Exception:
        pass

    nums = [int(z) for z in _int_pat.findall(s)]
    if len(nums) == 1:
        return {"kind": "exact", "lower": nums[0], "upper": nums[0], "raw": x}
    if len(nums) >= 2:
        a, b = nums[0], nums[1]
        if a == b:
            return {"kind": "exact", "lower": a, "upper": b, "raw": x}
        return {"kind": "range", "lower": min(a,b), "upper": max(a,b), "raw": x}

    return {"kind": "other", "lower": None, "upper": None, "raw": x}

def format_unknotting(lower, upper):
    if lower is None and upper is None:
        return None
    if lower is None or upper is None:
        return None
    return str([int(lower), int(upper)])



In [ ]:
# ----------------------------
# Load workbook and identify columns
# ----------------------------
df = pd.read_excel(XLSX_PATH)

knot_col  = pick_first_existing(df, ["knot_id", "name", "knot", "id"])
jones_col = pick_first_existing(df, ["jones_vector", "jones_polynomial_vector"])
pd_col    = pick_first_existing(df, ["pd_presentation", "pd_notation", "pd", "pd_code"])
u_col     = pick_first_existing(df, ["unknotting_number", "unknotting", "u"])

print("Columns:")
print("  knot_col :", knot_col)
print("  jones_col:", jones_col)
print("  pd_col   :", pd_col)
print("  u_col    :", u_col)

if knot_col is None or pd_col is None or u_col is None:
    raise ValueError("Could not identify the required knot / PD / unknotting-number columns.")

if jones_col is None:
    jones_col = "jones_vector"
    df[jones_col] = None
    print(f"Created missing Jones column: {jones_col}")

display(df.head())



In [ ]:
# ----------------------------
# Jones polynomial code from the attached notebook
# ----------------------------
def poly_add(p, q):
    r = dict(p)
    for e, c in q.items():
        r[e] = r.get(e, 0) + c
        if r[e] == 0:
            del r[e]
    return r

def poly_mul(p, q):
    r = {}
    for e1, c1 in p.items():
        for e2, c2 in q.items():
            e = e1 + e2
            r[e] = r.get(e, 0) + c1 * c2
    return {e:c for e,c in r.items() if c != 0}

def poly_monom(exp, coeff=1):
    return {int(exp): int(coeff)}

def poly_scale(p, s):
    return {e: c*s for e,c in p.items() if c*s != 0}

class DSU:
    def __init__(self):
        self.p = {}
    def find(self, x):
        if x not in self.p:
            self.p[x] = x
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.p[rb] = ra
    def n_components(self):
        return len({self.find(x) for x in self.p})

def bracket_from_pd(pd):
    n = len(pd)
    Delta = poly_add(poly_scale(poly_monom(2), -1), poly_scale(poly_monom(-2), -1))

    labels = set()
    for (a,b,c,d) in pd:
        labels.update([a,b,c,d])

    total = {}
    for mask in range(1 << n):
        dsu = DSU()
        for x in labels:
            dsu.find(x)

        a_count = 0
        b_count = 0
        for i, (a,b,c,d) in enumerate(pd):
            if ((mask >> i) & 1) == 0:
                a_count += 1
                dsu.union(a, b)
                dsu.union(c, d)
            else:
                b_count += 1
                dsu.union(b, c)
                dsu.union(d, a)

        loops = dsu.n_components()
        mon = poly_monom(a_count - b_count, 1)

        factor = {0: 1}
        for _ in range(loops - 1):
            factor = poly_mul(factor, Delta)

        total = poly_add(total, poly_mul(mon, factor))
    return total

def crossing_sign_pd(quad):
    a,b,c,d = quad
    return 1 if (a < c) == (b < d) else -1

def jones_string_from_pd(pd_list):
    if not pd_list:
        return None, "Empty PD"
    pdq = [tuple(map(int, q)) for q in pd_list]
    try:
        br = bracket_from_pd(pdq)
        w = sum(crossing_sign_pd(q) for q in pdq)
        sign = -1 if ((-3*w) % 2) else 1
        norm = poly_scale(poly_monom(-3*w, 1), sign)
        normed = poly_mul(norm, br)

        jt = {}
        for eA, c in normed.items():
            eT = Fraction(-eA, 4)
            jt[eT] = jt.get(eT, 0) + c
        jt = {e:c for e,c in jt.items() if c != 0}

        terms = []
        for e in sorted(jt.keys(), reverse=True):
            if e.denominator != 1:
                raise ValueError(f"Non-integral exponent encountered: {e}")
            c = jt[e]
            k = e.numerator
            if k == 0:
                mon = ""
            elif k == 1:
                mon = "t"
            else:
                mon = f"t^{k}"
            if mon == "":
                term = f"{c}"
            else:
                if c == 1:
                    term = mon
                elif c == -1:
                    term = "-" + mon
                else:
                    term = f"{c}*{mon}"
            terms.append(term)

        if not terms:
            return "0", None

        s = terms[0]
        for t in terms[1:]:
            if t.startswith("-"):
                s += " - " + t[1:]
            else:
                s += " + " + t
        return s, None
    except Exception as e:
        return None, repr(e)

def parse_jones_string_to_dict(jstr):
    if jstr is None:
        return None
    s = str(jstr).strip()
    if s == "" or s == "0":
        return {}

    s = s.replace(" - ", " + -")
    parts = [p.strip() for p in s.split(" + ") if p.strip()]
    poly = {}
    for term in parts:
        term = term.replace(" ", "")
        if "*t" in term:
            c_str, mon = term.split("*", 1)
            coeff = int(c_str)
        elif term.startswith("t") or term.startswith("-t"):
            coeff = -1 if term.startswith("-t") else 1
            mon = term[1:] if term.startswith("-t") else term
        else:
            coeff = int(term)
            exp = 0
            poly[exp] = poly.get(exp, 0) + coeff
            continue

        if mon == "t":
            exp = 1
        elif mon.startswith("t^"):
            exp = int(mon[2:])
        else:
            raise ValueError(f"Bad monomial format: {mon}")
        poly[exp] = poly.get(exp, 0) + coeff
    return {e:c for e,c in poly.items() if c != 0}

def poly_dict_to_knotinfo_vector(poly):
    if poly is None:
        return None
    if len(poly) == 0:
        return [0, 0, 0]
    mn, mx = min(poly.keys()), max(poly.keys())
    coeffs = [int(poly.get(e, 0)) for e in range(mn, mx + 1)]
    return [int(mn), int(mx)] + coeffs

def jones_vector_from_pd(pd_list):
    jstr, err = jones_string_from_pd(pd_list)
    if err is not None:
        return None, err
    poly = parse_jones_string_to_dict(jstr)
    vec = poly_dict_to_knotinfo_vector(poly)
    return vec, None



In [ ]:
# ----------------------------
# Fill missing Jones vectors in the database itself if needed
# ----------------------------
missing_before = 0
filled = 0
errors = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Filling missing Jones vectors"):
    vec = parse_vector_cell(row.get(jones_col))
    if ensure_minmax_coeffs(vec) is not None:
        continue

    missing_before += 1
    pd_list = parse_pd_cell(row.get(pd_col))
    if pd_list is None:
        errors.append((idx, row.get(knot_col), "missing/bad PD"))
        continue

    if len(pd_list) > 20:
        # the direct bracket expansion is exponential in crossing number
        # so we skip very large PDs here; the improvement pipeline only
        # needs Jones vectors for rows that are actually matched later.
        continue

    new_vec, err = jones_vector_from_pd(pd_list)
    if new_vec is not None:
        df.at[idx, jones_col] = str(new_vec)
        filled += 1
    else:
        errors.append((idx, row.get(knot_col), err))

print("Missing Jones entries before:", missing_before)
print("Filled directly from PD:", filled)
print("Unfilled / errors:", len(errors))
if errors[:10]:
    print("Sample errors:", errors[:10])



In [ ]:
# ----------------------------
# Build Jones lookup from the workbook
# The lookup stores every matching knot. The dependency pass below first
# takes the MAX inside one ambiguous Jones class, then the MIN across
# different dependency classes found by the search.
# ----------------------------
lookup = defaultdict(list)

for idx, row in df.iterrows():
    vec = parse_vector_cell(row.get(jones_col))
    parsed = ensure_minmax_coeffs(vec)
    if parsed is None:
        continue

    uk = parse_unknotting_entry(row.get(u_col))
    if uk["upper"] is None:
        continue

    mn, mx, coeffs = parsed
    sp = span_abs(mn, mx)
    rec = {
        "row_index": int(idx),
        "knot": row.get(knot_col),
        "upper": int(uk["upper"]),
        "lower": None if uk["lower"] is None else int(uk["lower"]),
        "mirror": False,
    }
    lookup[(sp, canon_coeff_key(coeffs))].append({**rec, "mirror": False})
    lookup[(sp, canon_coeff_key_mirror(coeffs))].append({**rec, "mirror": True})

print("Lookup keys:", len(lookup))



In [ ]:
# ----------------------------
# Training / RL utilities, adapted from the uploaded notebook
# ----------------------------
import io
import gzip

# Candidate local model files
MODEL_PATH_CANDIDATES = [
    MODELS_DIR / "best_model.zip",
    MODELS_DIR / "ppo_knot_rl_spherogram_continued.zip",
    OUT_DIR / "best_model.zip",
]

# Optional local training files
LOCAL_EXTRA_FILES = [
    TRAINING_DIR / "hard_unknots.csv",
    TRAINING_DIR / "very_hard_unknots.csv",
    TRAINING_DIR / "random_diagrams.csv",
    TRAINING_DIR / "random_diagrams.txt",
    DATA_DIR / "hard_unknots.csv",
    DATA_DIR / "very_hard_unknots.csv",
]


def make_sb3_load_custom_objects(default_lr: float = 3e-4):
    """
    Provides fallback objects for models saved under a different Python/cloudpickle
    stack where SB3 cannot deserialize callable schedules (e.g. lr_schedule).
    """
    lr_value = float(default_lr)

    def _lr_schedule(_progress_remaining: float) -> float:
        return lr_value

    return {
        "learning_rate": lr_value,
        "lr_schedule": _lr_schedule,
    }

_RE_DT_PREFIX = re.compile(r'^\s*DT\s*:\s*\[', re.I)
_RE_PDLIST    = re.compile(r'^\s*\[\s*(\[\s*\d+(?:\s*,\s*\d+){3}\s*\]\s*,?\s*)+\]\s*$')
_RE_XPD       = re.compile(r'[Xx]\s*\[')

def parse_link_strict(s: str) -> Link:
    t = s.strip()
    if _RE_DT_PREFIX.match(t):
        return Link(t)
    if _RE_PDLIST.match(t):
        try:
            pd_obj = json.loads(t)
        except json.JSONDecodeError:
            pd_obj = ast.literal_eval(t)
        return Link(pd_obj)
    if _RE_XPD.search(t):
        try:
            return Link(t)
        except Exception:
            items = re.findall(r'[Xx]\s*\[([^\]]+)\]', t)
            if not items:
                raise
            blocks = []
            for it in items:
                nums = [int(x.strip()) for x in it.split(',')]
                if len(nums) != 4:
                    raise ValueError("PD block must have 4 integers")
                blocks.append(nums)
            return Link(str(blocks))
    if (t.startswith("{") or t.startswith("[")) and not _RE_PDLIST.match(t):
        try:
            obj = json.loads(t)
            if isinstance(obj, dict):
                for key in ("pd", "PD", "pd_code", "PD_code", "dt", "DT"):
                    if key in obj:
                        return parse_link_strict(obj[key])
        except Exception:
            pass
    raise ValueError("Not a PD/DT code")

def clean_pd_lines(lines: Iterable[str], max_keep: int | None = None) -> List[str]:
    good = []
    for s in lines:
        try:
            _ = parse_link_strict(s)
            good.append(s.strip())
            if max_keep and len(good) >= max_keep:
                break
        except Exception:
            continue
    return good

def crossings(link: Link) -> int:
    return len(link.crossings)

def is_trivial_zero(link: Link) -> bool:
    return crossings(link) == 0

def riii_shuffle_only_link(link: Link, k: int, tries_per_move: int = 20):
    from spherogram.links import simplify as _simp
    list_fn  = getattr(_simp, "possible_type_III_moves", None)
    apply_fn = getattr(_simp, "reidemeister_III", None)
    if list_fn is None or apply_fn is None:
        return link, 0

    L = link
    done = 0
    for _ in range(k):
        moves = list_fn(L)
        if not moves:
            break
        tries = min(tries_per_move, len(moves))
        c0 = crossings(L)
        success = False
        for tri in random.sample(moves, tries):
            apply_fn(L, tri)
            if crossings(L) == c0:
                success = True
                break
        if not success:
            break
        done += 1
    return L, done

def read_first_col_local(path: str, has_header: bool = True, encoding: str = "utf-8") -> list[str]:
    out = []
    with open(path, "r", encoding=encoding, newline="") as f:
        rdr = csv.reader(f)
        if has_header:
            next(rdr, None)
        for row in rdr:
            if row:
                out.append(row[0].strip())
    return out

def workbook_pd_lines(max_keep: int | None = None) -> list[str]:
    raw = []
    for _, row in df.iterrows():
        pd_list = parse_pd_cell(row.get(pd_col))
        if pd_list is not None:
            raw.append(json.dumps(pd_list))
            if max_keep is not None and len(raw) >= max_keep:
                break
    return raw

@dataclass
class EnvCfg:
    max_steps: int = UNKNOTTER_MAX_STEPS
    step_penalty: float = 0.05
    reward_finish: float = 10.0
    allow_backtrack: bool = True
    cap_max: int = 8
    w_delta: float = 1.0
    w_uphill: float = 0.5
    w_potential: float = 0.02

class SphKnotEnv(gym.Env):
    def __init__(self, pd_lines: list[str], cfg: EnvCfg):
        super().__init__()
        self.cfg = cfg
        self.pd_lines = pd_lines
        self.rng = random.Random(SEED)
        self.num_actions = 4 if self.cfg.allow_backtrack else 3
        self.action_space = spaces.MultiDiscrete(
            np.array([self.num_actions, self.cfg.cap_max + 1], dtype=np.int64)
        )
        self.obs_dim = 6
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(self.obs_dim,), dtype=np.float32
        )
        self.L = None
        self._steps = 0
        self._last_drop = 0
        self._after_backtrack = False
        self._blocked = [False, False, False, False]

    def _reset_blocks(self):
        self._blocked = [False, False, False, False]

    def _map_blocked_mode(self, mode: int) -> int:
        m = mode % self.num_actions
        for _ in range(self.num_actions):
            if not self._blocked[m]:
                return m
            m = (m + 1) % self.num_actions
        return min(3, self.num_actions - 1)

    def _obs(self):
        c = crossings(self.L)
        try:
            comps = len(self.L.link_components)
        except Exception:
            comps = 1
        tmp = Link(self.L.PD_code())
        try:
            reduced = tmp.simplify(mode="basic")
        except TypeError:
            reduced = tmp.simplify()
        can_reduce = 1.0 if (reduced and crossings(tmp) < c) else 0.0
        recent = 1.0 if getattr(self, "_last_drop", 0) > 0 else 0.0
        return np.array([c, comps, self._steps, can_reduce, recent, 1.0], dtype=np.float32)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self._steps = 0
        self._last_drop = 0
        self._after_backtrack = False
        self._reset_blocks()
        for _ in range(10):
            s = self.rng.choice(self.pd_lines)
            try:
                self.L = parse_link_strict(s)
                break
            except Exception:
                self.L = None
        if self.L is None:
            self.L = parse_link_strict(self.pd_lines[0])
        return self._obs(), {"crossings": crossings(self.L)}

    def step(self, action):
        self._steps += 1
        if isinstance(action, (list, tuple, np.ndarray)):
            mode_req, cap = int(action[0]), int(action[1])
        else:
            mode_req, cap = int(action), 0
        cap = max(0, min(cap, self.cfg.cap_max))
        mode = self._map_blocked_mode(mode_req)

        c_before = crossings(self.L)
        if mode == 0:
            try:
                self.L.simplify(mode="basic")
            except TypeError:
                self.L.simplify()
        elif mode == 1:
            steps = (cap if cap > 0 else 1)
            self.L.simplify(mode="level", type_III_limit=steps)
        elif mode == 2:
            steps = (cap if cap > 0 else 1)
            self.L.simplify(mode="pickup", type_III_limit=steps)
        elif mode == 3 and self.num_actions == 4:
            steps = (cap if cap > 0 else 1)
            self.L.backtrack(steps=steps, prob_type_1=0.35, prob_type_2=0.65)
            self.L, _ = riii_shuffle_only_link(self.L, min(steps, 2))

        c_after = crossings(self.L)
        delta = c_before - c_after
        self._last_drop = max(delta, 0)

        reward = (
            self.cfg.w_delta * delta
            - self.cfg.w_uphill * max(0, -delta)
            - self.cfg.w_potential * c_after
            - self.cfg.step_penalty
        )

        done = False
        if is_trivial_zero(self.L):
            reward += self.cfg.reward_finish
            done = True
        if self._steps >= self.cfg.max_steps:
            done = True

        if delta > 0:
            self._reset_blocks()
        else:
            if mode == 3:
                self._reset_blocks()
            elif delta < 0:
                self._blocked[mode] = True

        self._after_backtrack = (mode == 3 and self.num_actions == 4)

        info = {
            "crossings": c_after,
            "delta": delta,
            "mode_requested": mode_req,
            "mode_effective": mode,
            "cap": cap,
            "blocked": tuple(self._blocked),
        }
        return self._obs(), reward, done, False, info

def load_training_pd_lines():
    raw = []

    for path in LOCAL_EXTRA_FILES:
        if path.exists():
            try:
                extra = read_first_col_local(str(path), has_header=True)
                raw += extra
                print(f"Loaded local extra {path.name}: {len(extra)}")
            except Exception as e:
                print(f"Could not load {path.name}: {e}")

    if not raw:
        raw = workbook_pd_lines()
        print(f"Falling back to workbook PD data: {len(raw)} examples")

    pd_lines = clean_pd_lines(raw, max_keep=None)
    random.Random(SEED).shuffle(pd_lines)
    if not pd_lines:
        raise RuntimeError(
            "No valid PD/DT strings available for training. "
            "Add local CSV/TXT files under training_data/ or ensure unknotting.xlsx contains PD data."
        )
    return pd_lines

def make_single_env(pd_list, cfg: EnvCfg):
    pd_str = json.dumps(pd_list)
    pd_lines_single = [pd_str]
    def _make():
        return SphKnotEnv(pd_lines_single, cfg)
    return DummyVecEnv([_make])

def run_unknotter_on_pd(pd_list, model, cfg: EnvCfg, episodes: int = 3, return_best_pd: bool = False):
    vec_env = make_single_env(pd_list, cfg)
    success = False
    best_crossings_global = len(pd_list)
    best_pd_global = [list(q) for q in pd_list]

    for _ in range(episodes):
        obs = vec_env.reset()
        best_crossings_ep = best_crossings_global
        best_pd_ep = best_pd_global

        for _step in range(cfg.max_steps):
            action, _ = model.predict(obs, deterministic=True)
            obs, rewards, dones, infos = vec_env.step(action)
            info = infos[0]
            cr = info.get("crossings", None)
            try:
                cur_pd = [list(q) for q in vec_env.envs[0].L.PD_code()]
            except Exception:
                cur_pd = None

            if cr is not None and cr < best_crossings_ep and cur_pd is not None:
                best_crossings_ep = cr
                best_pd_ep = cur_pd

            if cr == 0:
                success = True
                if cur_pd is not None:
                    best_crossings_ep = 0
                    best_pd_ep = cur_pd
                break
            if dones[0]:
                break

        if best_crossings_ep < best_crossings_global:
            best_crossings_global = best_crossings_ep
            best_pd_global = best_pd_ep
        if success:
            break

    vec_env.close()
    if return_best_pd:
        return success, best_crossings_global, best_pd_global
    return success, best_crossings_global



In [ ]:
# ----------------------------
# Load or train PPO model
# ----------------------------
cfg = EnvCfg(max_steps=UNKNOTTER_MAX_STEPS, allow_backtrack=True)

# Robust fallback in case the config cell was edited or executed out of order.
if "MODEL_PATH_CANDIDATES" not in globals():
    MODEL_PATH_CANDIDATES = [
        BASE / "best_model.zip",
        BASE / "ppo_knot_rl_spherogram_continued.zip",
        OUT_DIR / "best_model.zip",
    ]

best_model_path = None
for path in MODEL_PATH_CANDIDATES:
    if Path(path).exists():
        best_model_path = Path(path)
        break

if best_model_path is not None:
    custom_objects = make_sb3_load_custom_objects()
    model = PPO.load(str(best_model_path), device="auto", custom_objects=custom_objects)
    print("Loaded existing model:", best_model_path)
else:
    if not TRAIN_IF_MODEL_MISSING:
        raise FileNotFoundError(
            "No trained model found in MODEL_PATH_CANDIDATES, and TRAIN_IF_MODEL_MISSING=False."
        )
    print("No existing model found. Training a fresh one.")
    print("Searched these locations:")
    for p in MODEL_PATH_CANDIDATES:
        print("  -", p)

    pd_lines_train = load_training_pd_lines()
    vec_env = DummyVecEnv([lambda: SphKnotEnv(pd_lines_train, cfg)])
    model = PPO(
        "MlpPolicy",
        vec_env,
        learning_rate=3e-4,
        n_steps=2048,
        batch_size=256,
        n_epochs=10,
        gamma=0.995,
        gae_lambda=0.97,
        clip_range=0.2,
        ent_coef=0.01,
        vf_coef=0.5,
        max_grad_norm=0.5,
        seed=SEED,
        verbose=1,
    )
    model.learn(total_timesteps=TRAIN_STEPS_IF_NEEDED, progress_bar=True)
    best_model_path = OUT_DIR / "best_model.zip"
    model.save(str(best_model_path))
    vec_env.close()
    print("Saved trained model to:", best_model_path)

ACTIVE_MODEL_PATH = Path(best_model_path)
print("Active model path:", ACTIVE_MODEL_PATH)



In [ ]:
# ----------------------------
# Connect-sum, flip, and matching helpers
# ----------------------------
def flip_crossing_quad(quad):
    a, b, c, d = quad
    return [b, c, d, a]

def generate_single_flip_variants(pd_list):
    out = []
    n = len(pd_list)
    for i in range(n):
        flipped = []
        for j, quad in enumerate(pd_list):
            flipped.append(flip_crossing_quad(quad) if i == j else list(quad))
        out.append((i, flipped))
    return out

def pd_from_link(link_obj):
    return [list(q) for q in link_obj.PD_code()]

def connect_sum_pd(pd_left, pd_right):
    left = Link([list(q) for q in pd_left])
    right = Link([list(q) for q in pd_right])

    # Try several API shapes because spherogram versions differ.
    if hasattr(left, "connected_sum") and callable(getattr(left, "connected_sum")):
        out = left.connected_sum(right)
        if out is None:
            out = left
        return pd_from_link(out), None

    if hasattr(left, "connect_sum") and callable(getattr(left, "connect_sum")):
        out = left.connect_sum(right)
        if out is None:
            out = left
        return pd_from_link(out), None

    try:
        out = left + right
        return pd_from_link(out), None
    except Exception as e:
        return None, f"connect_sum_not_available: {e!r}"

def match_jones_vector_to_database(vec):
    parsed = ensure_minmax_coeffs(vec)
    if parsed is None:
        return []
    mn, mx, coeffs = parsed
    sp = span_abs(mn, mx)
    return lookup.get((sp, canon_coeff_key(coeffs)), [])

def best_upper_bound_from_matches(matches):
    if not matches:
        return None
    uppers = [m["upper"] for m in matches if m.get("upper") is not None]
    if not uppers:
        return None
    return max(uppers)

def normalize_knot_id(knot):
    return str(knot).strip()

def current_bounds_for_row(row_index):
    if row_index is None or row_index not in df.index:
        return {"kind": "missing", "lower": None, "upper": None, "raw": None}
    return parse_unknotting_entry(df.at[row_index, u_col])

def current_upper_for_match(match):
    row_index = match.get("row_index")
    bounds = current_bounds_for_row(row_index)
    if bounds["upper"] is not None:
        return int(bounds["upper"])
    if match.get("upper") is not None:
        return int(match["upper"])
    return None

def safe_dependency_matches_from_jones_matches(matches, source_knot=None):
    source_key = None if source_knot is None else normalize_knot_id(source_knot)
    usable = []
    for match in matches:
        knot_key = normalize_knot_id(match.get("knot"))
        if source_key is not None and knot_key == source_key:
            continue
        upper = current_upper_for_match(match)
        if upper is None:
            continue
        usable.append((int(upper), match))
    if not usable:
        return []
    safe_upper = max(upper for upper, _match in usable)
    return [match for upper, match in usable if upper == safe_upper]

def dependency_sort_key(dep):
    upper = dep.get("upper")
    return (10**9 if upper is None else int(upper), normalize_knot_id(dep.get("knot")))

def add_dependency(dependency_map, match, context, source_knot):
    dep_knot = normalize_knot_id(match.get("knot"))
    if dep_knot == normalize_knot_id(source_knot):
        return None

    row_index = match.get("row_index")
    bounds = current_bounds_for_row(row_index)
    upper = bounds.get("upper")
    lower = bounds.get("lower")
    if upper is None:
        upper = match.get("upper")
    if lower is None:
        lower = match.get("lower")
    if upper is None:
        return None

    rec = dependency_map.get(dep_knot)
    if rec is None:
        rec = {
            "knot": dep_knot,
            "row_index": None if row_index is None else int(row_index),
            "lower": None if lower is None else int(lower),
            "upper": int(upper),
            "raw_unknotting": None if bounds.get("raw") is None else str(bounds.get("raw")),
            "evidence_count": 0,
            "evidence": [],
        }
        dependency_map[dep_knot] = rec
    else:
        rec["lower"] = None if lower is None else int(lower)
        rec["upper"] = int(upper)
        rec["raw_unknotting"] = None if bounds.get("raw") is None else str(bounds.get("raw"))

    rec["evidence_count"] += 1
    if len(rec["evidence"]) < MAX_DEPENDENCY_EVIDENCE_PER_KNOT:
        ev = dict(context)
        ev["mirror"] = bool(match.get("mirror", False))
        rec["evidence"].append(ev)
    return rec


In [ ]:
# ----------------------------
# Build connect-sum source cohorts and experiment cases
# ----------------------------
_knot_order_pat = re.compile(r"^(\d+)(?:[a-zA-Z])?_(\d+)$")

def knot_crossing_and_number(knot_name):
    m = _knot_order_pat.match(str(knot_name).strip())
    if not m:
        return None
    return int(m.group(1)), int(m.group(2))

exact_10 = []
unknown_10 = []
exact_8 = []
five_rows = {}

for idx, row in df.iterrows():
    knot_name = normalize_knot_id(row.get(knot_col))
    order = knot_crossing_and_number(knot_name)
    if order is None:
        continue
    crossing_number, knot_num = order
    uk = parse_unknotting_entry(row.get(u_col))
    rec = {
        "row_index": int(idx),
        "knot": knot_name,
        "crossing": int(crossing_number),
        "knot_number": int(knot_num),
        "lower": uk.get("lower"),
        "upper": uk.get("upper"),
        "kind": uk.get("kind"),
    }
    if crossing_number == 10:
        if uk["kind"] == "exact":
            exact_10.append(rec)
        else:
            unknown_10.append(rec)
    if crossing_number == 8 and uk["kind"] == "exact":
        exact_8.append(rec)
    if crossing_number == 5 and knot_name in set(HARD_FIVE_CROSSING_KNOT_IDS):
        five_rows[knot_name] = rec

exact_10.sort(key=lambda r: r["knot_number"])
unknown_10.sort(key=lambda r: r["knot_number"])
exact_8.sort(key=lambda r: r["knot_number"])

missing_fives = [k for k in HARD_FIVE_CROSSING_KNOT_IDS if k not in five_rows]
if missing_fives:
    raise ValueError(f"Missing required 5-crossing knots in workbook: {missing_fives}")

trefoil_rows = [r for r in exact_10 + unknown_10 + exact_8 if r["knot"] == HARD_TREFOIL_KNOT_ID]
if not trefoil_rows:
    # Trefoil may not be in these three cohorts; locate globally.
    trefoil_idx = df.index[df[knot_col].astype(str).str.strip() == HARD_TREFOIL_KNOT_ID].tolist()
    if not trefoil_idx:
        raise ValueError(f"Trefoil knot {HARD_TREFOIL_KNOT_ID} not found in workbook")
    tidx = int(trefoil_idx[0])
    trefoil_row = {"row_index": tidx, "knot": HARD_TREFOIL_KNOT_ID}
else:
    trefoil_row = trefoil_rows[0]

print("10-crossing exact known:", len(exact_10))
print("10-crossing non-exact (unknown/range):", len(unknown_10))
print("8-crossing exact known:", len(exact_8))
print("5-crossing anchors:", sorted(five_rows))

experiment_cases = []
if RUN_TREFOIL_WITH_10_EXACT:
    for rec in exact_10:
        experiment_cases.append({
            "family": "trefoil_x_10_exact",
            "left_knot": HARD_TREFOIL_KNOT_ID,
            "right_knot": rec["knot"],
            "left_row_index": int(trefoil_row["row_index"]),
            "right_row_index": int(rec["row_index"]),
        })

if RUN_FIVE_WITH_8_EXACT:
    for five_id in HARD_FIVE_CROSSING_KNOT_IDS:
        for rec in exact_8:
            experiment_cases.append({
                "family": "five_x_8_exact",
                "left_knot": five_id,
                "right_knot": rec["knot"],
                "left_row_index": int(five_rows[five_id]["row_index"]),
                "right_row_index": int(rec["row_index"]),
            })

if MAX_CASES_PER_FAMILY is not None:
    by_family = defaultdict(list)
    for case in experiment_cases:
        by_family[case["family"]].append(case)
    trimmed = []
    for fam, fam_cases in by_family.items():
        trimmed.extend(fam_cases[:MAX_CASES_PER_FAMILY])
    experiment_cases = trimmed

print("Total experiment cases:", len(experiment_cases))
if experiment_cases:
    display(pd.DataFrame(experiment_cases).head(20))


In [ ]:
# ----------------------------
# Main connect-sum experiment loop (with dependency propagation)
# ----------------------------
def json_safe(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, Path):
        return str(obj)
    raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")

def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2, default=json_safe)

def append_jsonl(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(payload, ensure_ascii=False, default=json_safe) + "\n")

if OVERWRITE_EXPERIMENT_OUTPUTS:
    cleanup_paths = [RESULTS_JSONL_PATH, SUMMARY_JSON_PATH]
    if OVERWRITE_DEPENDENCY_OUTPUTS:
        cleanup_paths.extend([
            ALL_DEPENDENCIES_JSON_PATH,
            UNIMPROVED_DEPENDENCIES_JSON_PATH,
            PROPAGATION_RESULTS_JSON_PATH,
        ])
    for path in cleanup_paths:
        p = Path(path)
        if p.exists():
            p.unlink()

manifest = {
    "created_at": RUN_STAMP,
    "source_workbook": str(SOURCE_XLSX_PATH),
    "copied_workbook": str(XLSX_PATH),
    "seed": int(SEED),
    "trefoil": HARD_TREFOIL_KNOT_ID,
    "five_crossing_knots": list(HARD_FIVE_CROSSING_KNOT_IDS),
    "require_preflip_13": bool(REQUIRE_PREFLIP_13),
    "require_postflip_13": bool(REQUIRE_POSTFLIP_13),
    "max_cases_per_family": MAX_CASES_PER_FAMILY,
    "max_flips_per_case": MAX_FLIPS_PER_CASE,
    "dependency_propagation_enabled": bool(ENABLE_DEPENDENCY_PROPAGATION),
    "propagation_max_passes": MAX_PROPAGATION_PASSES,
    "counts": {
        "exact_10": len(exact_10),
        "unknown_10": len(unknown_10),
        "exact_8": len(exact_8),
        "cases": len(experiment_cases),
    },
}
write_json(RUN_MANIFEST_PATH, manifest)

results = []
family_stats = defaultdict(lambda: {"cases": 0, "flips": 0, "kept_after_filter": 0, "matched": 0})

timestamp = RUN_STAMP
case_nodes_for_propagation = []

for cnum, case in enumerate(experiment_cases, start=1):
    family = case["family"]
    family_stats[family]["cases"] += 1

    source_case_key = f"case_{cnum}:{normalize_knot_id(case['left_knot'])}#{normalize_knot_id(case['right_knot'])}"
    left_pd = parse_pd_cell(df.at[case["left_row_index"], pd_col])
    right_pd = parse_pd_cell(df.at[case["right_row_index"], pd_col])
    if left_pd is None or right_pd is None:
        row = {
            "case_index": cnum,
            "case_key": source_case_key,
            "family": family,
            "left_knot": case["left_knot"],
            "right_knot": case["right_knot"],
            "status": "missing_pd",
        }
        results.append(row)
        append_jsonl(RESULTS_JSONL_PATH, row)
        continue

    sum_pd, cs_err = connect_sum_pd(left_pd, right_pd)
    if sum_pd is None:
        row = {
            "case_index": cnum,
            "case_key": source_case_key,
            "family": family,
            "left_knot": case["left_knot"],
            "right_knot": case["right_knot"],
            "status": "connect_sum_failed",
            "error": cs_err,
        }
        results.append(row)
        append_jsonl(RESULTS_JSONL_PATH, row)
        continue

    preflip_crossings = len(sum_pd)
    if REQUIRE_PREFLIP_13 and preflip_crossings != 13:
        row = {
            "case_index": cnum,
            "case_key": source_case_key,
            "family": family,
            "left_knot": case["left_knot"],
            "right_knot": case["right_knot"],
            "status": "filtered_preflip_crossings",
            "preflip_crossings": preflip_crossings,
        }
        results.append(row)
        append_jsonl(RESULTS_JSONL_PATH, row)
        continue

    flip_variants = generate_single_flip_variants(sum_pd)
    if MAX_FLIPS_PER_CASE is not None:
        flip_variants = flip_variants[:MAX_FLIPS_PER_CASE]

    dependency_map = {}
    best_case_upper = None

    for flip_index, flipped_pd in flip_variants:
        family_stats[family]["flips"] += 1
        postflip_crossings = len(flipped_pd)
        if REQUIRE_POSTFLIP_13 and postflip_crossings != 13:
            continue

        family_stats[family]["kept_after_filter"] += 1
        success, min_crossings_found, best_pd = run_unknotter_on_pd(
            flipped_pd,
            model,
            cfg,
            episodes=UNKNOTTER_EPISODES_PER_FLIP,
            return_best_pd=True,
        )

        reduced_crossings = None if best_pd is None else len(best_pd)
        vec = None
        match_count = 0
        match_knots = []
        matched_upper = None
        candidate_upper = None
        status = "ran_no_match"

        if best_pd is not None and MIN_DATABASE_CROSSINGS <= reduced_crossings <= MAX_DATABASE_CROSSINGS:
            vec, err = jones_vector_from_pd(best_pd)
            if vec is not None:
                matches = match_jones_vector_to_database(vec)
                match_count = len(matches)
                if matches:
                    family_stats[family]["matched"] += 1
                    status = "matched"
                    match_knots = sorted({str(m.get("knot")) for m in matches})

                    context = {
                        "case_index": int(cnum),
                        "flip_index": int(flip_index),
                        "rl_success": bool(success),
                        "min_crossings_found": int(min_crossings_found),
                        "best_pd_crossings": int(reduced_crossings),
                        "jones_vector": vec,
                    }
                    safe_matches = safe_dependency_matches_from_jones_matches(matches, source_knot=source_case_key)
                    deps_for_this_run = []
                    for match in safe_matches:
                        rec = add_dependency(dependency_map, match, context, source_knot=source_case_key)
                        if rec is not None:
                            deps_for_this_run.append(rec)

                    if deps_for_this_run:
                        matched_upper = max(dep["upper"] for dep in deps_for_this_run if dep.get("upper") is not None)
                        candidate_upper = int(matched_upper) + 1
                        if best_case_upper is None or candidate_upper < int(best_case_upper):
                            best_case_upper = int(candidate_upper)

        row = {
            "case_index": cnum,
            "case_key": source_case_key,
            "family": family,
            "left_knot": case["left_knot"],
            "right_knot": case["right_knot"],
            "flip_index": int(flip_index),
            "status": status,
            "preflip_crossings": int(preflip_crossings),
            "postflip_crossings": int(postflip_crossings),
            "rl_success": bool(success),
            "min_crossings_found": int(min_crossings_found),
            "reduced_crossings": reduced_crossings,
            "jones_vector": vec,
            "match_count": int(match_count),
            "match_knots": match_knots,
            "matched_upper": matched_upper,
            "candidate_upper": candidate_upper,
        }
        results.append(row)
        append_jsonl(RESULTS_JSONL_PATH, row)

    dependencies = sorted(dependency_map.values(), key=dependency_sort_key)
    best_dependency_upper = min([d["upper"] for d in dependencies], default=None)
    dependency_candidate_upper = None if best_dependency_upper is None else int(best_dependency_upper) + 1

    case_nodes_for_propagation.append({
        "case_index": int(cnum),
        "case_key": source_case_key,
        "family": family,
        "left_knot": normalize_knot_id(case["left_knot"]),
        "right_knot": normalize_knot_id(case["right_knot"]),
        "direct_upper": best_case_upper,
        "current_upper": best_case_upper,
        "dependency_candidate_upper": dependency_candidate_upper,
        "dependencies": dependencies,
    })

all_payload = {
    "created_at": timestamp,
    "source_workbook": str(SOURCE_XLSX_PATH),
    "copied_workbook": str(XLSX_PATH),
    "results": results,
    "case_nodes": case_nodes_for_propagation,
}
write_json(ALL_DEPENDENCIES_JSON_PATH, all_payload)


# ----------------------------
# Dependency propagation over case nodes
# ----------------------------
def dependency_current_upper(dep, case_upper_by_key):
    dep_knot = normalize_knot_id(dep.get("knot"))
    if dep_knot in case_upper_by_key and case_upper_by_key[dep_knot] is not None:
        return int(case_upper_by_key[dep_knot])
    dep_upper = dep.get("upper")
    if dep_upper is None:
        return None
    return int(dep_upper)

def best_dependency_candidate(rec, case_upper_by_key):
    dep_bounds = []
    for dep in rec.get("dependencies", []):
        dep_upper = dependency_current_upper(dep, case_upper_by_key)
        if dep_upper is None:
            continue
        dep_bounds.append({
            "knot": normalize_knot_id(dep.get("knot")),
            "upper": int(dep_upper),
        })
    best_dep = min(dep_bounds, key=lambda d: (d["upper"], d["knot"]), default=None)
    candidate_upper = None if best_dep is None else int(best_dep["upper"]) + 1
    return candidate_upper, best_dep, sorted(dep_bounds, key=lambda d: (d["upper"], d["knot"]))

case_upper_by_key = {}
for rec in case_nodes_for_propagation:
    case_upper_by_key[rec["case_key"]] = rec.get("current_upper")

unimproved_case_nodes = [
    rec for rec in case_nodes_for_propagation
    if rec.get("direct_upper") is None and rec.get("dependency_candidate_upper") is not None
]

dependents_by_dependency = defaultdict(set)
for rec in case_nodes_for_propagation:
    src_case = rec.get("case_key")
    for dep in rec.get("dependencies", []):
        dep_knot = normalize_knot_id(dep.get("knot"))
        if dep_knot:
            dependents_by_dependency[dep_knot].add(src_case)

propagation_events = []
updated_by_propagation = defaultdict(list)
case_nodes_by_key = {rec["case_key"]: rec for rec in case_nodes_for_propagation}
pending = {rec["case_key"] for rec in case_nodes_for_propagation if rec.get("dependencies")}
pass_num = 0

if ENABLE_DEPENDENCY_PROPAGATION:
    while pending:
        if MAX_PROPAGATION_PASSES is not None and pass_num >= MAX_PROPAGATION_PASSES:
            break

        pass_num += 1
        current_batch = sorted(pending)
        pending = set()
        pass_updates = 0

        for case_key in current_batch:
            rec = case_nodes_by_key.get(case_key)
            if rec is None:
                continue

            current_upper = rec.get("current_upper")
            candidate_upper, best_dep, dep_bounds = best_dependency_candidate(rec, case_upper_by_key)
            should_update = (
                candidate_upper is not None and
                (current_upper is None or int(candidate_upper) < int(current_upper))
            )
            if not should_update:
                continue

            old_upper = current_upper
            rec["current_upper"] = int(candidate_upper)
            case_upper_by_key[case_key] = int(candidate_upper)
            pass_updates += 1

            event = {
                "pass": int(pass_num),
                "case_key": case_key,
                "case_index": int(rec["case_index"]),
                "old_upper": None if old_upper is None else int(old_upper),
                "new_upper": int(candidate_upper),
                "best_dependency": best_dep,
                "dependency_bounds_now": dep_bounds,
            }
            propagation_events.append(event)
            updated_by_propagation[case_key].append(event)

            for dependent in dependents_by_dependency.get(case_key, set()):
                if dependent != case_key:
                    pending.add(dependent)

        if pass_updates == 0:
            break

propagation_results = []
for rec in case_nodes_for_propagation:
    case_key = rec["case_key"]
    candidate_upper, best_dep, dep_bounds = best_dependency_candidate(rec, case_upper_by_key)
    propagation_results.append({
        "case_key": case_key,
        "case_index": int(rec["case_index"]),
        "family": rec["family"],
        "left_knot": rec["left_knot"],
        "right_knot": rec["right_knot"],
        "direct_upper": rec.get("direct_upper"),
        "final_upper": rec.get("current_upper"),
        "next_candidate_upper": candidate_upper,
        "updated_by_propagation": case_key in updated_by_propagation,
        "events": updated_by_propagation.get(case_key, []),
        "best_dependency_now": best_dep,
        "dependency_bounds_now": dep_bounds,
    })

write_json(UNIMPROVED_DEPENDENCIES_JSON_PATH, {
    "created_at": timestamp,
    "source_workbook": str(SOURCE_XLSX_PATH),
    "copied_workbook": str(XLSX_PATH),
    "nodes": unimproved_case_nodes,
})

write_json(PROPAGATION_RESULTS_JSON_PATH, {
    "created_at": timestamp,
    "source_workbook": str(SOURCE_XLSX_PATH),
    "copied_workbook": str(XLSX_PATH),
    "propagation_enabled": bool(ENABLE_DEPENDENCY_PROPAGATION),
    "propagation_max_passes": MAX_PROPAGATION_PASSES,
    "propagation_passes_run": int(pass_num),
    "propagation_updates": len(propagation_events),
    "events": propagation_events,
    "results": propagation_results,
})

summary = {
    "created_at": RUN_STAMP,
    "source_workbook": str(SOURCE_XLSX_PATH),
    "copied_workbook": str(XLSX_PATH),
    "result_count": len(results),
    "families": {k: v for k, v in family_stats.items()},
    "dependency_nodes": len(case_nodes_for_propagation),
    "propagation_updates": len(propagation_events),
    "propagation_passes_run": int(pass_num),
}
write_json(SUMMARY_JSON_PATH, summary)

print("Run manifest:", RUN_MANIFEST_PATH)
print("Per-trial JSONL:", RESULTS_JSONL_PATH)
print("Summary JSON:", SUMMARY_JSON_PATH)
print("All dependency JSON:", ALL_DEPENDENCIES_JSON_PATH)
print("Unimproved dependency JSON:", UNIMPROVED_DEPENDENCIES_JSON_PATH)
print("Propagation results JSON:", PROPAGATION_RESULTS_JSON_PATH)
display(pd.DataFrame(results).head(30))


Dependency propagation notebook for all configured targets.
